# 53 - Encode the scaled corpus with SimLM

Different text handling from every other encoder here: `26_baseline_simlm.ipynb` builds a separate title (company name) and body (everything else, same fields as `rich_text` minus name) and tokenizes them as a pair with `text_pair=`, not one combined string. CLS-token pooling, `intfloat/simlm-base-msmarco-finetuned`, a BERT-base-sized model (fast, not a 7B model like notebooks 47/51/52), so this uses the simpler chunked-checkpoint pattern rather than the aggressive time-budget one.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from dotenv import load_dotenv
from transformers import AutoTokenizer, AutoModel

load_dotenv()

RESULT_DIR = Path("result/53_encode_simlm_scaled")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = RESULT_DIR / "company_embeddings_checkpoint.npy"
FINAL_PATH = RESULT_DIR / "company_embeddings.npy"
CHUNK_SIZE = 20_000

combined = pd.read_parquet("result/44_build_scaled_corpus/combined_pool.parquet")


def build_title(row):
    name = row.get("name", "")
    return name.strip() if isinstance(name, str) and name.strip() else "Unknown company"


def build_body(row):
    """Same fields as every other baseline's rich text, minus name (that's the separate title here)."""
    parts = []
    for field, prefix in [
        ("country",           "Country:"),
        ("state",             "State:"),
        ("municipality",      "City:"),
        ("district",          "District:"),
        ("organization_type", "Type:"),
        ("organization_size", "Size:"),
        ("nace_code",         "Industry:"),
        ("summary",           ""),
    ]:
        val = row.get(field, "")
        if isinstance(val, str) and val.strip():
            parts.append(f"{prefix} {val}".strip() if prefix else val)
    kw = row.get("summary_keywords", "")
    if isinstance(kw, str) and kw.strip():
        kw_clean = kw.replace("'", "").replace("[", "").replace("]", "")
        parts.append(f"Keywords: {kw_clean}")
    return " | ".join(parts)


print("[Load] Building title + body fields for each company...")
titles = [build_title(row) for _, row in combined.iterrows()]
bodies = [build_body(row) for _, row in combined.iterrows()]
print(f"[Load] Companies to encode: {len(titles):,}")

print(f"[GPU] CUDA available : {torch.cuda.is_available()}")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"[GPU] Device : {torch.cuda.get_device_name(0)}")

print("[Encode] Loading SimLM (intfloat/simlm-base-msmarco-finetuned)...")
t0 = time.time()
REPO = "intfloat/simlm-base-msmarco-finetuned"
# Try local cache first with local_files_only -- avoids a slow/unauthenticated HF Hub network
# call hanging the job even when the model is already fully cached (same fix as notebooks 48-50).
try:
    tokenizer = AutoTokenizer.from_pretrained(REPO, local_files_only=True)
    model = AutoModel.from_pretrained(REPO, local_files_only=True).to(DEVICE)
    print("[Encode] Loaded from local cache -- skipped Hugging Face Hub network calls")
except Exception as e:
    print(f"[Encode] Not fully cached locally yet ({type(e).__name__}) -- retrying with network access (this will be slower)")
    tokenizer = AutoTokenizer.from_pretrained(REPO)
    model = AutoModel.from_pretrained(REPO).to(DEVICE)
model.eval()
print(f"[Encode] Model loaded in {time.time()-t0:.1f}s on {DEVICE}")


def cls_pool(last_hidden_state):
    emb = last_hidden_state[:, 0, :]
    return torch.nn.functional.normalize(emb, p=2, dim=1)

In [ ]:
@torch.no_grad()
def encode_passages(titles_batch, bodies_batch, batch_size=128):
    all_embs = []
    for i in range(0, len(titles_batch), batch_size):
        t_batch = titles_batch[i:i + batch_size]
        b_batch = bodies_batch[i:i + batch_size]
        encoded = tokenizer(t_batch, text_pair=b_batch, max_length=144, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
        outputs = model(**encoded)
        embs = cls_pool(outputs.last_hidden_state)
        all_embs.append(embs.cpu().float().numpy())
    return np.concatenate(all_embs, axis=0)


CHUNKS_DIR = RESULT_DIR / "chunks"
CHUNKS_DIR.mkdir(parents=True, exist_ok=True)


def atomic_save_npy(arr, path):
    """Write to a temp file then atomically rename -- a plain np.save() left a truncated,
    corrupted checkpoint on notebook 51 when a job was killed mid-write. Same fix applied here."""
    p = Path(path)
    tmp_path = p.with_suffix(".tmp" + p.suffix)
    np.save(tmp_path, arr)
    os.replace(tmp_path, path)


def get_chunk_files():
    return sorted(CHUNKS_DIR.glob("chunk_*.npy"), key=lambda p: int(p.stem.split("_")[1]))


if FINAL_PATH.exists() and np.load(FINAL_PATH, mmap_mode="r").shape[0] == len(titles):
    print("[Encode] Final embeddings already on disk -- skipping")
    embeddings = np.load(FINAL_PATH)
else:
    chunk_files = get_chunk_files()
    start = sum(np.load(f, mmap_mode="r").shape[0] for f in chunk_files)

    # One-time migration: an older run may have left a single ever-growing checkpoint file
    # (the pattern that triggered a bwUniCluster high-I/O warning -- 746GB written against only
    # 43GB of actual data, since it rewrote the full accumulated array on every chunk). Fold
    # whatever it already has into the new per-chunk format once, instead of re-encoding it.
    if CHECKPOINT_PATH.exists() and start == 0:
        legacy = np.load(CHECKPOINT_PATH)
        print(f"[Encode] Migrating legacy checkpoint ({legacy.shape[0]:,} rows) into per-chunk format...")
        atomic_save_npy(legacy, CHUNKS_DIR / f"chunk_{0:09d}.npy")
        start = legacy.shape[0]
        CHECKPOINT_PATH.unlink()
        chunk_files = get_chunk_files()

    if start:
        print(f"[Encode] Resuming -- {start:,}/{len(titles):,} already encoded across {len(chunk_files)} chunk files")

    t0 = time.time()
    for chunk_start in range(start, len(titles), CHUNK_SIZE):
        chunk_embs = encode_passages(
            titles[chunk_start:chunk_start + CHUNK_SIZE],
            bodies[chunk_start:chunk_start + CHUNK_SIZE],
            batch_size=128,
        )
        # Save ONLY this chunk as its own small file -- never rewrite everything already saved.
        atomic_save_npy(chunk_embs, CHUNKS_DIR / f"chunk_{chunk_start:09d}.npy")
        done_so_far = chunk_start + len(chunk_embs)
        elapsed = time.time() - t0
        print(f"[Encode] {done_so_far:,}/{len(titles):,} encoded ({elapsed/60:.1f} min elapsed)")

    # Assemble the final array exactly once, only now that every chunk is done.
    chunk_files = get_chunk_files()
    embeddings = np.concatenate([np.load(f) for f in chunk_files], axis=0)
    atomic_save_npy(embeddings, FINAL_PATH)
    for f in chunk_files:
        f.unlink()
    print(f"[Encode] Done. Embeddings shape: {embeddings.shape}")
    print(f"[Encode] Saved -> {FINAL_PATH}")